### Gradio Day!
Today we will build User Interfaces using the outrageously simple Gradio framework.

Prepare for joy!

Please note: your Gradio screens may appear in 'dark mode' or 'light mode' depending on your computer settings.

In [21]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [22]:
import gradio as gr # oh yeah

In [23]:
load_dotenv(override= True)

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and it begins with {openai_api_key[:8]}....")
else:
    print("OpenAI API Key is not set.")


if groq_api_key:
    print(f"Groq API Key exists and begins with {groq_api_key[:8]}....")
else:
    print("Groq API Key is not set")


if google_api_key:
    print(f"Google API Key exists and begins with {google_api_key[:8]}....")
else:
    print("Google API Key is not set")

OpenAI API Key exists and it begins with sk-proj-....
Groq API Key exists and begins with gsk_PvVX....
Google API Key exists and begins with AIzaSyB-....


In [60]:
openai_client = OpenAI()

groq_base_url = "https://api.groq.com/openai/v1"
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

groq = OpenAI(base_url= groq_base_url, api_key= groq_api_key)
gemini = OpenAI(base_url= gemini_base_url, api_key= google_api_key)

In [25]:
# Let's wrap a call to GPT-4.1-mini in a simple function

system_message = "You are a helpful assistant."

def message_gpt(prompt):
    messages= [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    response = openai_client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages= messages
    )

    return response.choices[0].message.content

In [26]:
# This can reveal the "training cut off", or the most recent date in the training data

message_gpt("What is today's date?")

"Today's date is June 8, 2024."

### User Interface time!

In [27]:
# here's a simple function

def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [28]:
shout("hello")

Shout has been called with input hello


'HELLO'

In [59]:
# gr.Interface(fn= shout, inputs= "textbox", outputs= "textbox", flagging_mode= "never").launch(inbrowser= True)

In [30]:
# Adding hare=True means that it can be accessed publically
# A more permanent hosting is available using a platform called Spaces from HuggingFace, which we will touch on next week.
# NOTE: Some Anti-virus software and Corporate Firewalls not like you using share=True.
# If you're at work on on a work network, I suggest skip this test.

# gr.Interface(fn= shout, inputs= "textbox", outputs= "text_box", flagging_mode= "never").launch(share= True)

In [35]:
# Adding-inbrowser= True opens up a new browser window automatically

# gr.Interface(fn= shout, inputs= "textbox", outputs= "textbox", flagging_mode= "never").launch(inbrowser= True)

### Adding authentication

Gradio makes it very easy to have userids and passwords.  
Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [36]:
# gr.Interface(fn= shout, inputs= "textbox", outputs= "textbox", flagging_mode="never").launch(inbrowser= True, auth= (os.getenv("GRADIO_USERNAME"), os.getenv("GRADIO_PASSWORD")))

### Forcing dark mode

Gradio appears in light mode or dark mode depending on the settings of the browser and computer. There is a way to force gradio to appear in dark mode, but Gradio recommends against this as it should be a user preference (particularly for accessibility reasons). But if you wish to force dark mode for your screens, below is how to do it.

In [33]:
# Define this variable and then pass js=force_dark_mode when creating the Interface.

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
# gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(js=force_dark_mode)

In [38]:
# And now - changing the function from "shout" to "message_gpt"

message_input= gr.Textbox(label="Your message: ", info= "Enter ta message to be shouted", lines= 7)
message_output = gr.Textbox(label= "Response", lines=8)

view = gr.Interface(
    fn= shout,
    title= "Shout",
    inputs= [message_input],
    outputs= [message_output],
    examples= ["hello", "howdy"],
    flagging_mode= "never"
)

# view.launch()

In [58]:
# And now - changing the function from "shout" to "message_gpt"

message_input= gr.Textbox(label="Your message: ", info="Enter a message fot gpt-4.1-mini", lines=7)
message_output= gr.Textbox(label= "Your response: ", lines= 8)

view = gr.Interface(
    fn= message_gpt,
    title= "GPT",
    inputs= [message_input],
    outputs= [message_output],
    examples= ["hello", "howdy"],
    flagging_mode= "never"
)

# view.launch()

In [57]:
# Let's use Markdown
# Are you wondering why it makes any difference to set system_message when it's not referred to in the code below it?
# I'm taking advantage of system_message being a global variable, used back in the message_gpt function (go take a look)
# Not a great software engineering practice, but quite common during Jupyter Lab R&D!

system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
# view.launch()

In [54]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the "yield" keyword),
# Please take a llook at the intermediate Python guide in the guides folder

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    response = openai_client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages= messages,
        stream= True
    )

    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [56]:
message_input = gr.Textbox(label= "Your message: ", info= "Enter your message for GPT-4.1-mini", lines= 7)
message_output = gr.Markdown(label= "Response: ")

view = gr.Interface(
    fn= stream_gpt,
    title= "GPT",
    inputs= [message_input],
    outputs= [message_output],
    examples= [
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode= "never"
)

# view.launch()

In [61]:
def stream_gemini(prompt):
    messages= [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    response = gemini.chat.completions.create(
        model= "gemini-2.5-flash-lite",
        messages= messages,
        stream= True
    )

    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content
        yield result

In [63]:
message_input = gr.Textbox(label= "Your message: ", info= "Enter a message for Gemini 2.5 flash lite", lines= 7)
message_output = gr.Markdown(label= "Response: ")

view = gr.Interface(
    fn= stream_gemini,
    title= "Gemini",
    inputs= message_input,
    outputs= message_output,
    examples= [
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode= "never"
)

# view.launch()

### And now getting fancy

Remember to check the Intermediate Python Guide if you're unsure about generators and "yield"

In [70]:
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [71]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Claude"]
        ], 
    flagging_mode="never"
    )

# view.launch()

### Building a company brochure generator
Now you know how - it's simple!